# Temporal ML & Validation Methodology — Bitcoin (BTC-USD)

**Goal:** Apply Machine Learning to predict Bitcoin daily returns, with a strong focus on **rigorous validation methodology** to avoid the most common pitfalls in financial ML.

**Key questions:**
- Can ML models reliably predict Bitcoin returns?
- How do we know if a good score is real — or just data leakage?
- How does model performance vary across different market regimes?

**Dataset:** BTC-USD daily data from Yahoo Finance (2018–2024)

## 1. Setup & Data Loading

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf

from statsmodels.tsa.arima.model import ARIMA
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.model_selection import train_test_split

np.random.seed(42)
plt.rcParams["figure.figsize"] = (12, 5)

In [ ]:
# Download Bitcoin daily data (2018–2024)
data = yf.download("BTC-USD", start="2018-01-01", end="2024-01-01", auto_adjust=False)
data = data[["Close", "Volume"]].dropna()

print(f"Dataset: {data.shape[0]} trading days")
data.head()

## 2. Exploratory Analysis — Prices & Returns

In [ ]:
price   = data["Close"]
returns = price.pct_change().dropna()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(price)
axes[0].set_title("BTC-USD Closing Price — Non-stationary")
axes[0].set_ylabel("Price ($)")
axes[0].grid(True)

axes[1].plot(returns)
axes[1].set_title("BTC-USD Daily Returns — Stationary, high volatility")
axes[1].set_ylabel("Return")
axes[1].grid(True)

plt.tight_layout()
plt.show()

print(f"Mean return: {float(returns.mean()):.4f}")
print(f"Std  return: {float(returns.std()):.4f}  (vs ~0.01 for SPY — Bitcoin is ~3-4x more volatile)")

In [ ]:
# Return distribution — fat tails are very pronounced on crypto
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

returns.hist(bins=80, ax=axes[0])
axes[0].set_title("Return Distribution — Heavy tails (leptokurtosis)")
axes[0].set_xlabel("Return")
axes[0].grid(True)

rolling_vol = returns.rolling(5).std()
axes[1].plot(rolling_vol)
axes[1].set_title("5-day Rolling Volatility — Clustering visible")
axes[1].set_ylabel("Volatility")
axes[1].grid(True)

plt.tight_layout()
plt.show()

## 3. Feature Engineering — Temporal Features

To use standard ML on a time series, we convert the past into explicit input features.
**Critical rule:** every feature must use only information available at prediction time — no look-ahead.

Features built:
- `lag_1, lag_2, lag_5`: past returns at different horizons
- `rolling_mean_5`: 5-day average return (trend proxy)
- `rolling_vol_5`: 5-day return volatility (short-term risk)
- `momentum_5`: price difference over 5 days (directional strength)
- `volume_change`: daily volume percentage change

In [ ]:
df = data.copy()
df["return"] = df["Close"].pct_change()

# Lagged returns — encode short-term autocorrelation
df["lag_1"] = df["return"].shift(1)
df["lag_2"] = df["return"].shift(2)
df["lag_5"] = df["return"].shift(5)

# Rolling statistics — encode recent trend and risk level
df["rolling_mean_5"] = df["return"].rolling(window=5).mean()
df["rolling_vol_5"]  = df["return"].rolling(window=5).std()

# Momentum — directional signal over 5 days
df["momentum_5"]    = df["Close"] - df["Close"].shift(5)

# Volume activity — spike in volume often precedes large moves
df["volume_change"] = df["Volume"].pct_change()

df = df.dropna()
print(f"Final dataset: {df.shape[0]} rows × {df.shape[1]} columns")
df.head()

## 4. Target Variable & Train/Test Split

**Target:** next-day return `y_t = r_{t+1}` (regression, not classification).

**Split strategy:** strict chronological split — test set is always *after* train set.
Using random shuffle on time series introduces look-ahead bias and invalidates evaluation.

In [ ]:
features = ["lag_1", "lag_2", "lag_5", "rolling_mean_5", "rolling_vol_5",
            "momentum_5", "volume_change"]

X = df[features]
y = df["return"].shift(-1).dropna()   # next-day return
X = X.iloc[:len(y)]                   # align lengths

split = int(len(X) * 0.8)
X_train, X_test = X.iloc[:split], X.iloc[split:]
y_train, y_test = y.iloc[:split], y.iloc[split:]

print(f"Train: {len(X_train)} days | Test: {len(X_test)} days")
print(f"Train: {X_train.index[0].date()} → {X_train.index[-1].date()}")
print(f"Test:  {X_test.index[0].date()}  → {X_test.index[-1].date()}")

## 5. ARIMA Baseline

In [ ]:
train_returns = df["return"].iloc[:split]
test_returns  = df["return"].iloc[split:split + len(y_test)]

arima_result = ARIMA(train_returns, order=(1, 0, 1)).fit()
pred_arima   = arima_result.forecast(steps=len(test_returns)).values

mse_arima = mean_squared_error(test_returns, pred_arima)
mae_arima = mean_absolute_error(test_returns, pred_arima)

print(f"ARIMA(1,0,1) — MSE: {mse_arima:.6f} | MAE: {mae_arima:.6f}")
print("(Forecasts will be near-flat — low autocorrelation in BTC returns)")

## 6. Random Forest Regressor

In [ ]:
model_rf = RandomForestRegressor(n_estimators=300, max_depth=5, random_state=42)
model_rf.fit(X_train, y_train)
pred_rf = model_rf.predict(X_test)

mse_rf = mean_squared_error(y_test, pred_rf)
mae_rf = mean_absolute_error(y_test, pred_rf)

print(f"Random Forest — MSE: {mse_rf:.6f} | MAE: {mae_rf:.6f}")

In [ ]:
# Feature importance — which past signals are most predictive?
importance = pd.Series(model_rf.feature_importances_, index=features).sort_values()
importance.plot(kind="barh", title="Random Forest — Feature Importance")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()
# rolling_vol_5 and momentum_5 typically rank highest:
# consistent with GARCH finding that volatility has memory,
# and with documented momentum effects on crypto markets

In [ ]:
# Visual comparison: actual vs. predicted returns
plt.figure(figsize=(12, 4))
plt.plot(y_test.values, label="Actual returns", alpha=0.7)
plt.plot(pred_rf, label="Random Forest predictions", alpha=0.9)
plt.title("Random Forest — Predictions vs. Actual Returns")
plt.xlabel("Observations")
plt.ylabel("Return")
plt.legend()
plt.grid(True)
plt.show()

## 7. Neural Network (MLP Regressor)

**Important:** StandardScaler must be fitted on training data only.
Fitting on the full dataset would leak test statistics into training — a subtle but critical form of data leakage.

In [ ]:
# Scale features — fit on train only, transform both
scaler = StandardScaler()
scaler.fit(X_train)
X_train_scaled = scaler.transform(X_train)
X_test_scaled  = scaler.transform(X_test)

model_mlp = MLPRegressor(
    hidden_layer_sizes=(64, 32),
    activation="relu",
    early_stopping=True,
    random_state=42,
    max_iter=500
)
model_mlp.fit(X_train_scaled, y_train)
pred_mlp = model_mlp.predict(X_test_scaled)

mse_mlp = mean_squared_error(y_test, pred_mlp)
mae_mlp = mean_absolute_error(y_test, pred_mlp)

print(f"MLP — MSE: {mse_mlp:.6f} | MAE: {mae_mlp:.6f}")

## 8. Model Comparison

In [ ]:
results = pd.DataFrame({
    "Model": ["ARIMA(1,0,1)", "Random Forest", "MLP (64-32)"],
    "MSE":   [mse_arima, mse_rf, mse_mlp],
    "MAE":   [mae_arima, mae_rf, mae_mlp]
})

print(results.to_string(index=False))

results.set_index("Model")[["MSE", "MAE"]].plot(
    kind="bar", figsize=(9, 4),
    title="Model Comparison — MSE and MAE on Test Set"
)
plt.xticks(rotation=0)
plt.grid(True)
plt.tight_layout()
plt.show()

## 9. Validation Deep-Dive — The Danger of Random Split

Using `shuffle=True` on time series data is a **critical methodological error**.
It leaks future observations into training, producing artificially optimistic scores.
The following cell demonstrates this quantitatively.

In [ ]:
# INTENTIONAL ERROR — for illustration purposes only
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X, y, test_size=0.2, shuffle=True, random_state=42
)

model_random = RandomForestRegressor(n_estimators=300, max_depth=5, random_state=42)
model_random.fit(X_train_r, y_train_r)
mse_random = mean_squared_error(y_test_r, model_random.predict(X_test_r))

print("=== Impact of random vs. temporal split ===")
print(f"MSE with RANDOM split  (wrong): {mse_random:.6f}")
print(f"MSE with TEMPORAL split (right): {mse_rf:.6f}")
print(f"Ratio: {mse_random/mse_rf:.2f}x  — random split artificially deflates error")
print()
print("Why? With random split, the model sees 2022 data while predicting 2019.")
print("Neighboring days in time share very similar features → artificially easy prediction.")

## 10. Rolling Window Validation

A single train/test split evaluates the model on one arbitrary time window.
**Rolling validation** trains and tests across multiple windows, revealing:
- Average performance across different market conditions
- Performance **variance** — how stable is the model over time?
- Which periods are systematically harder to predict?

In [ ]:
def rolling_validation_rf(X, y, train_size=400, test_size=50, step=50,
                           n_estimators=100, max_depth=5):
    """Walk-forward validation with fixed-size training window.
    At each step: train on [i : i+train_size], test on [i+train_size : i+train_size+test_size].
    Returns list of MSE scores, one per window.
    """
    scores = []
    n = len(X)

    for i in range(0, n - train_size - test_size, step):
        X_tr = X.iloc[i : i + train_size]
        y_tr = y.iloc[i : i + train_size]
        X_te = X.iloc[i + train_size : i + train_size + test_size]
        y_te = y.iloc[i + train_size : i + train_size + test_size]

        model = RandomForestRegressor(
            n_estimators=n_estimators, max_depth=max_depth, random_state=42
        )
        model.fit(X_tr, y_tr)
        scores.append(mean_squared_error(y_te, model.predict(X_te)))

    return scores


rolling_scores = rolling_validation_rf(X, y)
print(f"Number of validation windows: {len(rolling_scores)}")

In [ ]:
mean_score = np.mean(rolling_scores)
std_score  = np.std(rolling_scores)
cv         = std_score / mean_score

plt.figure(figsize=(12, 4))
plt.plot(rolling_scores, marker='o', markersize=4, label="MSE per window")
plt.axhline(mean_score, color='red', linestyle='--',
            label=f"Mean MSE: {mean_score:.6f}")
plt.fill_between(range(len(rolling_scores)),
                 mean_score - std_score, mean_score + std_score,
                 alpha=0.2, color='red', label=f"±1 std: {std_score:.6f}")
plt.title("Rolling Validation — MSE by Window\nHigh variance reveals regime instability")
plt.xlabel("Window index")
plt.ylabel("MSE")
plt.legend()
plt.grid(True)
plt.show()

print(f"Mean MSE:             {mean_score:.6f}")
print(f"Std MSE:              {std_score:.6f}")
print(f"Coefficient of variation: {cv:.1%}")
print()
print("High CV → model performance is highly regime-dependent.")
print("Periods of high volatility (crises) consistently produce higher MSE.")

## 11. Conclusion

**Key findings:**

- All models struggle to predict Bitcoin returns — the signal-to-noise ratio is fundamentally low (ρ(k) ≈ 0)
- Random Forest and MLP marginally outperform ARIMA by capturing non-linear interactions
- **Feature importance** confirms: `rolling_vol_5` and `momentum_5` are the most predictive — consistent with volatility clustering (TP2) and crypto momentum effects
- **Random split inflates performance** — the ratio vs. temporal split is a direct measure of data leakage
- **Rolling validation** reveals that performance is highly unstable across time windows — the model systematically fails during high-volatility regimes (COVID 2020, FTX collapse 2022)

**Validation rules for financial ML:**
1. Always split chronologically — never shuffle
2. Fit scalers/encoders on train only
3. Use rolling validation, not a single split, to get a reliable performance estimate
4. Report performance distribution (mean ± std), not a single number